# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, 'RecordSet' entities are tabular datasets or collections of records. Each RecordSet may have several 'field' definitions, each with a unique `@id` for reliable reference.

This section will print available RecordSets and their fields, displaying their `@id`s and names.

In [ ]:
from pprint import pprint

record_sets = metadata.record_set
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(record_sets)} RecordSet(s):\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        print("  Fields:")
        for field in rs.get('field', []):
            print(f"    - Field @id: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we demonstrate extracting all found record sets.

In [ ]:
# Collect all RecordSet @id's
if not metadata.record_set or len(metadata.record_set) == 0:
    print("No record sets available for extraction.")
else:
    record_set_ids = [rs['@id'] for rs in metadata.record_set]
    print("RecordSet IDs:", record_set_ids)
    dataframes = {}
    
    for record_set_id in record_set_ids:
        print(f"\nExtracting data from RecordSet: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if len(records):
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded {len(df)} rows.")
                print(f"Columns: {df.columns.tolist()}")
                display(df.head())
            else:
                print("No data records found.")
        except Exception as e:
            print(f"Error loading records for {record_set_id}: {str(e)}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

**Note:** Below is a generic example; adjust field `@id`s and thresholds according to the data overview in Section 2.

In [ ]:
# Choose a RecordSet and Field IDs for analysis (update these as necessary)
if not metadata.record_set or len(metadata.record_set) == 0:
    print("No record sets found for EDA.")
else:
    # Pick the first RecordSet for demonstration
    selected_record_set_id = metadata.record_set[0]['@id']
    selected_df = dataframes[selected_record_set_id] if selected_record_set_id in dataframes else None

    # List field @id's and select a numeric one for EDA
    fields = metadata.record_set[0].get('field', [])
    numeric_field_id = None
    print("Available fields in selected record set:")
    for f in fields:
        print(f"- {f.get('@id')}: name='{f.get('name')}', type={f.get('dataType')}")
        if numeric_field_id is None and f.get('dataType', '').lower() in ['float', 'number', 'integer', 'double']:
            numeric_field_id = f['@id']
    
    if not selected_df is None and numeric_field_id in selected_df.columns:
        print(f"\nRunning filtering and normalization on numeric field: {numeric_field_id}\n")
        threshold = selected_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(selected_df[numeric_field_id]) else 10
        filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):\n{filtered_df.head()}")
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Try grouping by another field (if available)
        group_field_id = None
        for f in fields:
            if f['@id'] != numeric_field_id and f['@id'] in selected_df.columns and pd.api.types.is_string_dtype(selected_df[f['@id']]):
                group_field_id = f['@id']
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric group field found for demonstration.")
    else:
        print("No suitable numeric field found in the first RecordSet for EDA. Please update numeric_field_id based on your data overview.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example of visualizing the distribution and relationships in a chosen numeric field. Edit field `@id`s as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not metadata.record_set or len(metadata.record_set) == 0:
    print("No record sets found for visualization.")
else:
    # Use the selected_df and numeric_field_id from the previous cell if present
    if not selected_df is None and numeric_field_id is not None and numeric_field_id in selected_df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(selected_df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
        
        # If grouping field is available
        if 'group_field_id' in locals() and group_field_id is not None and group_field_id in selected_df.columns:
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=selected_df[group_field_id], y=selected_df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric field available for visualization. Update field @id as appropriate.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- In this notebook, we used the `mlcroissant` library to load, inspect, and explore the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset (FAIR^2).
- We demonstrated how to systematically reference RecordSets and fields using their `@id`s in alignment with the Croissant schema.
- Exploratory analysis and visualizations were performed on numeric fields, which are crucial for statistical understanding of biological data.
- This notebook provides a strong starting point for more advanced domain-specific analyses or machine learning workflows on this dataset.